# Hybrid Encoding (BLOSUM + Physicochemical Features)

This notebook creates a **hybrid encoding** combining:
1. **BLOSUM62 encoding** (620 features) - Evolutionary relationships
2. **Physicochemical features** (35 features) - From notebook 02

## Why Hybrid?

✓ **Best of both worlds**: Sequence similarity + domain knowledge
✓ **Comprehensive**: Captures evolutionary AND biochemical patterns
✓ **Proven effective**: Your friend's research shows this works well
✓ **More features**: More information for the model to learn from

## What's Included?

### BLOSUM62 Features (620)
- Each position × 20 BLOSUM scores
- Captures amino acid substitution patterns

### Physicochemical Features (35)
- Amino acid composition (20 features)
- Hydrophobicity, charge, polarity
- Secondary structure propensity
- Cysteine-specific features
- And more from notebook 02

**Total**: ~655 features

## Configuration

In [1]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_FILE = "../data_engineered/train_with_features.csv"  # From notebook 02 (has physiochemical features)
OUTPUT_FILE = "../data_engineered/hybrid/train_with_features_hybrid.csv"

# Sequence parameters
MAX_LENGTH = 31  # All sequences are 31 amino acids

# Processing
CHUNK_SIZE = 10000  # Process in chunks to save memory

print("Configuration loaded:")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Chunk size: {CHUNK_SIZE}")

Configuration loaded:
  Input: ../data_engineered/train_with_features.csv
  Output: ../data_engineered/hybrid/train_with_features_hybrid.csv
  Max length: 31
  Chunk size: 10000


## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

✓ Libraries imported


## BLOSUM62 Matrix

In [3]:
# BLOSUM62 substitution matrix
BLOSUM62 = {
    'A': [ 4, -1, -2, -2,  0, -1, -1,  0, -2, -1, -1, -1, -1, -2, -1,  1,  0, -3, -2,  0],
    'R': [-1,  5,  0, -2, -3,  1,  0, -2,  0, -3, -2,  2, -1, -3, -2, -1, -1, -3, -2, -3],
    'N': [-2,  0,  6,  1, -3,  0,  0,  0,  1, -3, -3,  0, -2, -3, -2,  1,  0, -4, -2, -3],
    'D': [-2, -2,  1,  6, -3,  0,  2, -1, -1, -3, -4, -1, -3, -3, -1,  0, -1, -4, -3, -3],
    'C': [ 0, -3, -3, -3,  9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
    'Q': [-1,  1,  0,  0, -3,  5,  2, -2,  0, -3, -2,  1,  0, -3, -1,  0, -1, -2, -1, -2],
    'E': [-1,  0,  0,  2, -4,  2,  5, -2,  0, -3, -3,  1, -2, -3, -1,  0, -1, -3, -2, -2],
    'G': [ 0, -2,  0, -1, -3, -2, -2,  6, -2, -4, -4, -2, -3, -3, -2,  0, -2, -2, -3, -3],
    'H': [-2,  0,  1, -1, -3,  0,  0, -2,  8, -3, -3, -1, -2, -1, -2, -1, -2, -2,  2, -3],
    'I': [-1, -3, -3, -3, -1, -3, -3, -4, -3,  4,  2, -3,  1,  0, -3, -2, -1, -3, -1,  3],
    'L': [-1, -2, -3, -4, -1, -2, -3, -4, -3,  2,  4, -2,  2,  0, -3, -2, -1, -2, -1,  1],
    'K': [-1,  2,  0, -1, -3,  1,  1, -2, -1, -3, -2,  5, -1, -3, -1,  0, -1, -3, -2, -2],
    'M': [-1, -1, -2, -3, -1,  0, -2, -3, -2,  1,  2, -1,  5,  0, -2, -1, -1, -1, -1,  1],
    'F': [-2, -3, -3, -3, -2, -3, -3, -3, -1,  0,  0, -3,  0,  6, -4, -2, -2,  1,  3, -1],
    'P': [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4,  7, -1, -1, -4, -3, -2],
    'S': [ 1, -1,  1,  0, -1,  0,  0,  0, -1, -2, -2,  0, -1, -2, -1,  4,  1, -3, -2, -2],
    'T': [ 0, -1,  0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1,  1,  5, -2, -2,  0],
    'W': [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1,  1, -4, -3, -2, 11,  2, -3],
    'Y': [-2, -2, -2, -3, -2, -1, -2, -3,  2, -1, -1, -2, -1,  3, -3, -2, -2,  2,  7, -1],
    'V': [ 0, -3, -3, -3, -1, -2, -2, -3, -3,  3,  1, -2,  1, -1, -2, -2,  0, -3, -1,  4]
}

AA_ORDER = list("ARNDCQEGHILKMFPSTWYV")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

print("✓ BLOSUM62 matrix loaded")

✓ BLOSUM62 matrix loaded


## Hybrid Encoding Functions

In [4]:
def sequence_to_blosum(sequence, max_length=31):
    """
    Encode a single sequence using BLOSUM62.
    
    Returns:
        1D array of BLOSUM features (max_length * 20)
    """
    blosum_matrix = np.zeros((max_length, 20), dtype=np.float32)
    
    for i, aa in enumerate(sequence[:max_length]):
        if aa in BLOSUM62:
            blosum_matrix[i, :] = BLOSUM62[aa]
    
    return blosum_matrix.flatten()  # Shape: (620,)

def encode_chunk_hybrid(chunk_df, max_length=31):
    """
    Create hybrid encoding: BLOSUM + physicochemical features.
    
    Returns DataFrame with:
    - ID and label columns
    - Physicochemical features (from input)
    - BLOSUM features (newly encoded)
    """
    sequences = chunk_df['Sequence'].values
    
    # Encode all sequences with BLOSUM
    blosum_features = np.vstack([
        sequence_to_blosum(seq, max_length)
        for seq in sequences
    ])
    
    # Create BLOSUM column names
    blosum_cols = []
    for pos in range(max_length):
        for aa in AA_ORDER:
            blosum_cols.append(f"blosum_pos{pos}_{aa}")
    
    # Create DataFrame with BLOSUM features
    blosum_df = pd.DataFrame(blosum_features, columns=blosum_cols, index=chunk_df.index)
    
    # Identify physicochemical feature columns (exclude ID, Sequence, labels)
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    meta_cols = ['ID', 'Sequence']
    
    phys_cols = [col for col in chunk_df.columns 
                 if col not in meta_cols + label_cols]
    
    # Keep ID, labels, physicochemical features + add BLOSUM features
    result_df = pd.concat([
        chunk_df[['ID'] + label_cols + phys_cols],
        blosum_df
    ], axis=1)
    
    return result_df

print("✓ Hybrid encoding functions defined")

✓ Hybrid encoding functions defined


## Load and Encode Data

In [5]:
print("="*80)
print("LOADING AND ENCODING DATA")
print("="*80)

# Check if input file exists
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run 02_feature_engineering.ipynb first."
    )

# Get total rows
total_rows = sum(1 for _ in open(INPUT_FILE)) - 1  # -1 for header
print(f"\nTotal sequences to encode: {total_rows:,}")

# Process in chunks
chunks_processed = []
for chunk in tqdm(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), 
                   total=(total_rows // CHUNK_SIZE) + 1,
                   desc="Encoding chunks"):
    
    encoded_chunk = encode_chunk_hybrid(chunk, max_length=MAX_LENGTH)
    chunks_processed.append(encoded_chunk)

# Combine all chunks
print("\nCombining chunks...")
encoded_df = pd.concat(chunks_processed, ignore_index=True)

print(f"\n✓ Encoding complete!")
print(f"  Total samples: {len(encoded_df):,}")
print(f"  Total features: {len(encoded_df.columns)}")

LOADING AND ENCODING DATA

Total sequences to encode: 89,010


Encoding chunks: 100%|█████████████████████████████████████████████████████████| 9/9 [00:05<00:00,  1.65it/s]



Combining chunks...

✓ Encoding complete!
  Total samples: 89,010
  Total features: 658


## Analyze Feature Composition

In [6]:
print("\n" + "="*80)
print("FEATURE COMPOSITION ANALYSIS")
print("="*80)

# Count different feature types
label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
blosum_cols = [col for col in encoded_df.columns if col.startswith('blosum_')]
phys_cols = [col for col in encoded_df.columns 
             if col not in ['ID'] + label_cols + blosum_cols]

print(f"\nFeature breakdown:")
print(f"  BLOSUM features: {len(blosum_cols)}")
print(f"  Physicochemical features: {len(phys_cols)}")
print(f"  Total feature columns: {len(blosum_cols) + len(phys_cols)}")
print(f"  Label columns: {len(label_cols)}")
print(f"  ID column: 1")
print(f"  Grand total columns: {len(encoded_df.columns)}")

print(f"\nPhysicochemical features included:")
for i, col in enumerate(phys_cols[:10], 1):  # Show first 10
    print(f"  {i}. {col}")
if len(phys_cols) > 10:
    print(f"  ... and {len(phys_cols) - 10} more")


FEATURE COMPOSITION ANALYSIS

Feature breakdown:
  BLOSUM features: 620
  Physicochemical features: 34
  Total feature columns: 654
  Label columns: 3
  ID column: 1
  Grand total columns: 658

Physicochemical features included:
  1. aa_A
  2. aa_C
  3. aa_D
  4. aa_E
  5. aa_F
  6. aa_G
  7. aa_H
  8. aa_I
  9. aa_K
  10. aa_L
  ... and 24 more


## Verify Encoding

In [7]:
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

# Check for NaN/Inf values
nan_count = encoded_df.isnull().sum().sum()
inf_count = np.isinf(encoded_df.select_dtypes(include=[np.number])).sum().sum()

print(f"\nData quality:")
print(f"  NaN values: {nan_count}")
print(f"  Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print("  ⚠️  Warning: Found invalid values!")
else:
    print("  ✓ All values are valid")

# Check BLOSUM features
print(f"\nBLOSUM features:")
print(f"  Count: {len(blosum_cols)}")
print(f"  Min: {encoded_df[blosum_cols].min().min():.2f}")
print(f"  Max: {encoded_df[blosum_cols].max().max():.2f}")
print(f"  Mean: {encoded_df[blosum_cols].mean().mean():.2f}")

# Check physicochemical features
print(f"\nPhysicochemical features:")
print(f"  Count: {len(phys_cols)}")
if len(phys_cols) > 0:
    print(f"  Min: {encoded_df[phys_cols].min().min():.2f}")
    print(f"  Max: {encoded_df[phys_cols].max().max():.2f}")
    print(f"  Mean: {encoded_df[phys_cols].mean().mean():.2f}")


VERIFICATION

Data quality:
  NaN values: 0
  Inf values: 0
  ✓ All values are valid

BLOSUM features:
  Count: 620
  Min: -4.00
  Max: 11.00
  Mean: -1.08

Physicochemical features:
  Count: 34


UFuncTypeError: ufunc 'less_equal' did not contain a loop with signature matching types (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.StrDType'>) -> None

## Save Encoded Data

In [8]:
print("\n" + "="*80)
print("SAVING ENCODED DATA")
print("="*80)

# Create output directory
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Save to CSV
encoded_df.to_csv(OUTPUT_FILE, index=False)

file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB

print(f"\n✓ Saved to: {OUTPUT_FILE}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Samples: {len(encoded_df):,}")
print(f"  Total columns: {len(encoded_df.columns)}")
print(f"  Feature columns: {len(blosum_cols) + len(phys_cols)}")

print("\n" + "="*80)
print("✓ HYBRID ENCODING COMPLETE!")
print("="*80)
print(f"\nThis encoding combines:")
print(f"  • BLOSUM62 (evolutionary similarity)")
print(f"  • Physicochemical properties (domain knowledge)")
print(f"\nNext step: Run 04_train_val_split.ipynb with this file to create train/val splits.")


SAVING ENCODED DATA

✓ Saved to: ../data_engineered/hybrid/train_with_features_hybrid.csv
  File size: 281.3 MB
  Samples: 89,010
  Total columns: 658
  Feature columns: 654

✓ HYBRID ENCODING COMPLETE!

This encoding combines:
  • BLOSUM62 (evolutionary similarity)
  • Physicochemical properties (domain knowledge)

Next step: Run 04_train_val_split.ipynb with this file to create train/val splits.


## Why Hybrid is Powerful

### BLOSUM62 Contributes:
```
Position 15 has Cysteine:
- BLOSUM scores tell us C is similar to S (score +1)
- But very different from W (score -2)
- Model learns evolutionary patterns
```

### Physicochemical Features Contribute:
```
Sequence has:
- 20% hydrophobic amino acids
- Net charge of +2
- 3 cysteines
- High secondary structure propensity
- Model learns biochemical patterns
```

### Together:
- **BLOSUM**: What amino acids are present and how they relate
- **Physicochemical**: What properties those amino acids confer
- **Result**: More complete picture for the model to learn from

### Expected Performance:
Based on your friend's research, hybrid encoding typically:
- Outperforms one-hot encoding
- Matches or slightly exceeds BLOSUM-only
- Best all-around encoding for ensemble